# Math Prerequisites → Linear Algebra | Chapter 4: Special Matrices & Norms

> **Why this chapter exists:** PCA relies on specific types of matrices — orthogonal matrices (the PC loading matrix), diagonal matrices (eigenvalue matrix), and the Gram matrix (key for Dual PCA in high dimensions). The nuclear norm is the core of Robust PCA. Understanding these objects by name and property allows you to read PCA derivations fluently.

---

## 1. Orthogonal Matrices

A square matrix $Q \in \mathbb{R}^{n \times n}$ is **orthogonal** if its columns are orthonormal:
$$Q^T Q = Q Q^T = I \quad \Leftrightarrow \quad Q^{-1} = Q^T$$

### **Properties**
- **Free inverse:** $Q^{-1} = Q^T$ — the most computationally convenient inversion possible.
- **Preserves lengths:** $\|Q\vec{x}\|_2 = \|\vec{x}\|_2$ — rotation doesn't change vector length.
- **Preserves angles:** $(Q\vec{x}) \cdot (Q\vec{y}) = \vec{x} \cdot \vec{y}$ — rotation doesn't change angles.
- **Determinant = ±1:** $\det(Q) = +1$ (pure rotation) or $-1$ (rotation + reflection).

### **Why This Matters for PCA**
The PC loading matrix $W = [\vec{w}_1 | \cdots | \vec{w}_d]$ (columns = principal components) is orthogonal:
- $W^T W = I$ (orthonormal PCs)
- The transformation $Z = XW$ is a **rigid rotation** — it changes coordinates but not distances.
- Reconstruction is trivial: $X = ZW^T$ (since $W^{-1} = W^T$).
- This means PCA is **lossless** when you keep all $d$ components — zero reconstruction error.

---

## 2. Diagonal Matrices

A diagonal matrix has non-zero entries only on the main diagonal:
$$D = \text{diag}(d_1, d_2, ..., d_n) = \begin{bmatrix} d_1 & & \\ & \ddots & \\ & & d_n \end{bmatrix}$$

### **Properties**
- **Action:** $D\vec{x}$ scales each component independently: $(D\vec{x})_i = d_i x_i$.
- **Inverse:** $D^{-1} = \text{diag}(1/d_1, ..., 1/d_n)$ — trivial if all $d_i \neq 0$.
- **Powers:** $D^k = \text{diag}(d_1^k, ..., d_n^k)$.
- **Always symmetric:** $D = D^T$.

### **In PCA**
The eigenvalue matrix $\Lambda = \text{diag}(\lambda_1, ..., \lambda_d)$ is diagonal. This makes many PCA operations trivially cheap:
- **Whitening:** $\Lambda^{-1/2} = \text{diag}(1/\sqrt{\lambda_1}, ..., 1/\sqrt{\lambda_d})$
- **Reconstruction error:** $\sum_{i=k+1}^d \lambda_i = \text{tr}(\Lambda) - \sum_{i=1}^k \lambda_i$
- **PPCA noise estimate:** involves diagonal operations on $\Lambda$

---

## 3. Matrix Norms

Just as vector norms measure the "size" of a vector, matrix norms measure the "size" of a matrix.

### **Frobenius Norm**
$$\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2} = \sqrt{\text{tr}(A^T A)} = \sqrt{\sum_i \sigma_i^2}$$

The Frobenius norm is the entry-wise $L_2$ norm — treat the matrix as a long vector and take its $L_2$ norm. It equals the square root of the sum of squared singular values.

**PCA use:** The reconstruction error in PCA is measured in Frobenius norm:
$$\|X - \hat{X}\|_F^2 = \sum_{i=k+1}^r \sigma_i^2$$

### **Spectral Norm (Operator Norm)**
$$\|A\|_2 = \sigma_1 \quad \text{(the largest singular value)}$$

Represents the maximum stretching factor of the matrix — the largest it can stretch any unit vector.

**PCA use:** The Eckart-Young theorem also holds for the spectral norm.

### **Nuclear Norm (Trace Norm)**
$$\|A\|_* = \sum_i \sigma_i \quad \text{(sum of all singular values)}$$

The nuclear norm is the convex envelope of the rank function — minimizing the nuclear norm encourages **low-rank solutions**. This is the key idea in **Robust PCA** (Chapter 4 of the PCA series).

**Why nuclear norm ≈ rank?**
- Rank$(A) = $ number of non-zero singular values.
- Minimizing rank is NP-hard (combinatorial).
- Minimizing $\|A\|_*$ (convex!) promotes sparsity in singular values → encourages low-rank.
- This is the matrix analog of: $L_0$ norm (count non-zeros) → $L_1$ norm (convex, promotes sparsity).

### **Norm Summary Table**
| Norm | Formula | SVD Equivalent | Role in PCA |
| :--- | :--- | :--- | :--- |
| Frobenius | $\sqrt{\sum_{ij} A_{ij}^2}$ | $\sqrt{\sum_i \sigma_i^2}$ | Reconstruction error |
| Spectral | $\max_{\|x\|=1}\|Ax\|$ | $\sigma_1$ | Eckart-Young (spectral form) |
| Nuclear | $\text{tr}(\sqrt{A^T A})$ | $\sum_i \sigma_i$ | Robust PCA low-rank penalty |

---

## 4. Low-Rank Matrices

A matrix $A \in \mathbb{R}^{m \times n}$ is **rank-$k$** if it has exactly $k$ non-zero singular values ($k \ll \min(m,n)$).

### **The outer product form**
Any rank-$k$ matrix can be written as:
$$A = \sum_{i=1}^k \sigma_i \vec{u}_i \vec{v}_i^T = U_k \Sigma_k V_k^T$$

This is a sum of $k$ rank-1 matrices. Each rank-1 matrix $\vec{u}_i \vec{v}_i^T$ has structure — it's the outer product of two vectors.

### **Why Real-World Data is Approximately Low-Rank**
Real datasets have correlations — features don't vary independently. A dataset with $d=500$ features might effectively live in a 20-dimensional subspace. This means the covariance matrix $\Sigma \in \mathbb{R}^{500 \times 500}$ is approximately rank-20 — only 20 eigenvalues are meaningfully non-zero.

**PCA exploits this:** by keeping only $k \ll d$ components, PCA finds the best rank-$k$ approximation to the data, discarding the near-zero eigenvalue directions as noise or irrelevant variation.

### **Storage Efficiency**
- Full matrix: $m \times n$ values
- Rank-$k$ factored: $(m + n) \times k$ values
- Compression ratio: $\frac{mn}{(m+n)k}$

For a $1000 \times 1000$ matrix with rank 10: compression = $10^6 / (2 \times 10^4) = 50\times$.

---

## 5. The Gram Matrix: Key to Dual PCA

For a data matrix $X \in \mathbb{R}^{N \times d}$ (centered), we define two products:

| Name | Formula | Shape | Use |
| :--- | :--- | :---: | :--- |
| **Covariance matrix** | $\Sigma = X^T X / (N-1)$ | $d \times d$ | Standard PCA ($N > d$) |
| **Gram matrix** | $G = X X^T / (N-1)$ | $N \times N$ | Dual PCA ($N < d$) |

### **The Critical Shared Eigenvalue Property**

**Theorem:** Let $X = U\Sigma_s V^T$ (SVD). Then:
$$X^T X = V \Sigma_s^T \Sigma_s V^T \quad \Rightarrow \quad \text{eigenvalues of } X^T X = \sigma_i^2$$
$$X X^T = U \Sigma_s \Sigma_s^T U^T \quad \Rightarrow \quad \text{eigenvalues of } X X^T = \sigma_i^2$$

Both $X^T X$ and $X X^T$ share the same **non-zero eigenvalues** $\sigma_1^2, ..., \sigma_r^2$ where $r = \text{rank}(X)$.

**Consequence for Dual PCA:**
- When $d \gg N$: computing $X^T X \in \mathbb{R}^{d \times d}$ is infeasible (e.g., $d=50{,}000$: a 2.5 billion entry matrix).
- Instead, compute $G = XX^T \in \mathbb{R}^{N \times N}$ (much smaller when $N \ll d$).
- Eigendecompose $G$: $G = U_G \Lambda U_G^T$.
- Recover PCs: $V = X^T U_G \Lambda^{-1/2}$ (the PC directions in the original $d$-dimensional space).

This is the key derivation in Chapter 2 of the PCA series.

---

## 6. Implementation Playground

In [ ]:
# ─── CELL 1: Orthogonal Matrix Properties + Gram Matrix Shared Eigenvalues ─────
import numpy as np

# ── Orthogonal matrix: rotation by 30° ───────────────────────────────────────
theta = np.pi / 6  # 30 degrees
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

print("=== Orthogonal Matrix (Rotation 30°) ===")
print(f"Q^T Q = I? {np.allclose(Q.T @ Q, np.eye(2))}")
print(f"det(Q) = {np.linalg.det(Q):.4f}  (should be +1 for pure rotation)")

# Verify length preservation
x = np.array([3., 4.])
Qx = Q @ x
print(f"||x|| = {np.linalg.norm(x):.4f},  ||Qx|| = {np.linalg.norm(Qx):.4f}  (should be equal)")

# ── Gram matrix vs covariance: shared non-zero eigenvalues ───────────────────
print("\n=== Gram Matrix vs Covariance Matrix ===")
np.random.seed(0)
N, d = 8, 20  # N < d: high-dimensional setting
X = np.random.randn(N, d)
X = X - X.mean(axis=0)  # center

cov = X.T @ X           # d x d  (20x20)
gram = X @ X.T          # N x N  (8x8)

evals_cov = np.sort(np.linalg.eigvalsh(cov))[::-1]
evals_gram = np.sort(np.linalg.eigvalsh(gram))[::-1]

print(f"Covariance shape: {cov.shape}, Gram shape: {gram.shape}")
print(f"Non-zero eigenvalues of cov:  {evals_cov[:N].round(4)}")
print(f"Non-zero eigenvalues of gram: {evals_gram.round(4)}")
print(f"They match? {np.allclose(evals_cov[:N], evals_gram, atol=1e-8)}")
print(f"Number of zero eigenvalues in covariance (d-N = {d-N}): {np.sum(evals_cov < 1e-8)}")

In [ ]:
# ─── CELL 2: Matrix Norms — Frobenius, Spectral, Nuclear ──────────────────────
import numpy as np

# Toy matrix
A = np.array([[3., 1., 2.],
              [0., 2., 1.],
              [1., 0., 4.]])

U, s, Vt = np.linalg.svd(A)

# Frobenius norm
frobenius_direct  = np.sqrt(np.sum(A**2))
frobenius_via_svd = np.sqrt(np.sum(s**2))
frobenius_numpy   = np.linalg.norm(A, 'fro')
print(f"Frobenius norm: direct={frobenius_direct:.4f}, via SVD={frobenius_via_svd:.4f}, numpy={frobenius_numpy:.4f}")

# Spectral norm
spectral_svd   = s[0]  # largest singular value
spectral_numpy = np.linalg.norm(A, 2)
print(f"Spectral norm: σ₁={spectral_svd:.4f}, numpy={spectral_numpy:.4f}")

# Nuclear norm
nuclear = np.sum(s)
nuclear_numpy = np.linalg.norm(A, 'nuc')
print(f"Nuclear norm: Σσᵢ={nuclear:.4f}, numpy={nuclear_numpy:.4f}")

print(f"\nSingular values: {s.round(4)}")
print(f"Rank: {np.linalg.matrix_rank(A)}")

# Low-rank approximation and norm of error
print("\n=== Rank-k Approximation Errors ===")
for k in range(1, len(s)+1):
    A_k = (U[:, :k] * s[:k]) @ Vt[:k, :]
    frob_err = np.sqrt(np.sum(s[k:]**2))  # Eckart-Young prediction
    actual_err = np.linalg.norm(A - A_k, 'fro')
    print(f"k={k}: ||A - A_k||_F = {actual_err:.4f}  (Eckart-Young: sqrt(Σσ²_>k) = {frob_err:.4f})")